In [1]:
import os
import pandas as pd

paths = {}
base_path = r"C:\Users\kinga\Desktop\studia\DS\ED_project\fma_small\fma_small"
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".mp3"):
            track_id = int(file.split(".")[0])
            paths[track_id] = os.path.join(root, file)

tracks = pd.read_csv(r"C:\Users\kinga\Desktop\studia\DS\ED_project\tracks.csv", index_col=0, header=[0, 1])
keep_tracks = tracks[tracks[('set', 'subset')] == 'small']
df = keep_tracks[('track', 'genre_top')].reset_index()
df.columns = ['track_id', 'genre']

df['path'] = df['track_id'].map(paths)
df = df.dropna(subset=['path']).reset_index(drop=True)

genres = sorted(df['genre'].unique())
label2id = {label: i for i, label in enumerate(genres)}
id2label = {i: label for i, label in enumerate(genres)}
df['label'] = df['genre'].map(label2id)

print(f"Załadowano {len(df)} utworów z {len(genres)} gatunków.")

Załadowano 8000 utworów z 8 gatunków.


In [2]:
import numpy as np
import librosa
import cv2
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class SpectrogramGenerator(Sequence):
    def __init__(self, dataframe, batch_size=32, dim=(128, 128), n_channels=1, n_classes=8, shuffle=True):
        self.dataframe = dataframe
        self.batch_size = batch_size
        self.dim = dim
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_df = self.dataframe.iloc[index*self.batch_size : (index+1)*self.batch_size]
        
        X = np.empty((self.batch_size, *self.dim, self.n_channels))
        y = np.empty((self.batch_size), dtype=int)

        for i, (idx, row) in enumerate(batch_df.iterrows()):
            X[i,] = self._load_spectrogram(row['path'])
            y[i] = row['label']

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            self.dataframe = self.dataframe.sample(frac=1).reset_index(drop=True)

    def _load_spectrogram(self, path):
        try:
            y, sr = librosa.load(path, duration=10)
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            S_dB = librosa.power_to_db(S, ref=np.max)

            S_dB = (S_dB - S_dB.min()) / (S_dB.max() - S_dB.min())
            S_resized = cv2.resize(S_dB, self.dim)
            return S_resized.reshape(*self.dim, self.n_channels)
        except:
            return np.zeros((*self.dim, self.n_channels))

In [3]:
from tensorflow.keras import layers, models

def build_cnn_model(input_shape=(128, 128, 1), num_classes=8):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.BatchNormalization(),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.BatchNormalization(),
        
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_gen = SpectrogramGenerator(train_df, batch_size=32)
val_gen = SpectrogramGenerator(val_df, batch_size=32)

model = build_cnn_model()

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15
)

c:\Users\kinga\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\kinga\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
c:\Users\kinga\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_in

Epoch 1/15
 22/200 ━━━━━━━━━━━━━━━━━━━━ 9:30 3s/step - accuracy: 0.1537 - loss: 9.9621 

C:\Users\kinga\AppData\Local\Temp\ipykernel_6088\1103375921.py:42: RuntimeWarning: invalid value encountered in divide
  S_dB = (S_dB - S_dB.min()) / (S_dB.max() - S_dB.min())


 27/200 ━━━━━━━━━━━━━━━━━━━━ 9:21 3s/step - accuracy: 0.1566 - loss: 9.2034

KeyboardInterrupt: 